题目地址：https://tianchi.aliyun.com/competition/entrance/531993/information

运行环境Python 3.14.4t、numpy 2.4.4、scikit-learn 1.8.0、pandas 3.0.2

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import cross_val_score

In [2]:
train = pd.read_csv('train.csv')

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 22500 entries, 0 to 22499
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                22500 non-null  int64  
 1   age               22500 non-null  int64  
 2   job               22500 non-null  str    
 3   marital           22500 non-null  str    
 4   education         22500 non-null  str    
 5   default           22500 non-null  str    
 6   housing           22500 non-null  str    
 7   loan              22500 non-null  str    
 8   contact           22500 non-null  str    
 9   month             22500 non-null  str    
 10  day_of_week       22500 non-null  str    
 11  duration          22500 non-null  int64  
 12  campaign          22500 non-null  int64  
 13  pdays             22500 non-null  int64  
 14  previous          22500 non-null  int64  
 15  poutcome          22500 non-null  str    
 16  emp_var_rate      22500 non-null  float64
 17  cons

# 部分特征变量分析

## month字段分析

In [4]:
print('每个month的样本量：')
month_counts = train.groupby(['month']).agg(count=('id', 'count'))
print(month_counts)

month_subscribe_counts = train.groupby(['month', 'subscribe'], as_index=False).agg(count=('id', 'count'))
month_subscribe_counts = month_subscribe_counts[month_subscribe_counts['subscribe'] == 'yes']
month_subscribe_counts.set_index('month', inplace=True)

print('每个month中成功认购的比例：')
ratio = month_subscribe_counts['count'] / month_counts['count']
ratio

每个month的样本量：
       count
month       
apr     1510
aug     3340
dec      199
jul     3815
jun     2838
mar      426
may     7235
nov     2242
oct      494
sep      401
每个month中成功认购的比例：


month
apr    0.213245
aug    0.118862
dec    0.517588
jul    0.103014
jun    0.119803
mar    0.504695
may    0.069800
nov    0.113292
oct    0.459514
sep    0.488778
Name: count, dtype: float64

由此可见，month变量大概率对预测subscribe变量有价值。

## day_of_week字段分析

In [5]:
print('每个day_of_week的样本量：')
day_counts = train.groupby(['day_of_week']).agg(count=('id', 'count'))
print(day_counts)

day_subscribe_counts = train.groupby(['day_of_week', 'subscribe'], as_index=False).agg(count=('id', 'count'))
day_subscribe_counts = day_subscribe_counts[day_subscribe_counts['subscribe'] == 'yes']
day_subscribe_counts.set_index('day_of_week', inplace=True)
print(day_subscribe_counts)

print('每个month中成功认购的比例：')
ratio = day_subscribe_counts['count'] / day_counts['count']
ratio

每个day_of_week的样本量：
             count
day_of_week       
fri           4247
mon           4653
thu           4728
tue           4414
wed           4458
            subscribe  count
day_of_week                 
fri               yes    529
mon               yes    536
thu               yes    670
tue               yes    629
wed               yes    588
每个month中成功认购的比例：


day_of_week
fri    0.124559
mon    0.115194
thu    0.141709
tue    0.142501
wed    0.131898
Name: count, dtype: float64

由此可见，day_of_week变量大概率对预测subscribe变量没有价值。

# subscribe字段分析

In [6]:
(train['subscribe'] == 'yes').sum() / len(train['subscribe'])

np.float64(0.1312)

由此可见目标变量不平衡。预测正确率超过86.88%才算合格。

# 数据预处理

## 剔除部分变量

In [7]:
X_train = train.drop(columns=['id', 'subscribe', 'day_of_week'])
y_train = train['subscribe']

## 创建变量预处理器

In [8]:

categorical_features = X_train.select_dtypes(include='str').columns
numeric_features = X_train.select_dtypes(include='number').columns

num_OHE_processor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features)
    ]
)

num_OE_processor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OrdinalEncoder(), categorical_features)
    ]
)

OE_processor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(), categorical_features)
    ],
    remainder='passthrough'
)

OHE_processor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features)
    ],
    remainder='passthrough'
)

# 结果记录函数

In [9]:
results = pd.DataFrame({
    '估计器': [],
    '估计器实例': [],
    '平均交叉验证分数': [],
    '整个数据集的拟合时间': []
})


def record(model, grid_search):
    global results

    result = {
        '估计器': [model],
        '估计器实例': [grid_search.best_estimator_],
        '平均交叉验证分数': [grid_search.best_score_],
        '整个数据集的拟合时间': [grid_search.refit_time_]
    }

    print(f'平均交叉验证分数:{result['平均交叉验证分数']}')
    print(f'整个数据集的拟合时间:{result['整个数据集的拟合时间']}')

    temp = pd.DataFrame(result)
    results = pd.concat([results, temp], axis=0)


# LogisticRegression

In [10]:

clf = make_pipeline(num_OHE_processor, LogisticRegression(class_weight="balanced", random_state=0))

param_grid = [
    # 默认参数
    # {},
    {
        "logisticregression__solver": ["lbfgs"],
        "logisticregression__l1_ratio": [0.0],
        "logisticregression__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "logisticregression__solver": ["liblinear"],
        "logisticregression__l1_ratio": [1.0],
        "logisticregression__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "logisticregression__solver": ["saga"],
        "logisticregression__l1_ratio": [0.25, 0.5, 0.75],
        "logisticregression__C": [0.1, 1, 10],
    },
]

grid_search = GridSearchCV(clf, param_grid, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

record('LogisticRegression', grid_search)

# 默认参数：
# '平均交叉验证分数': np.float64(0.7861333333333334), '整个数据集的拟合时间': 0.09022760391235352
# 最佳参数：
# '平均交叉验证分数': np.float64(0.7863111111111111), '整个数据集的拟合时间': 0.5209152698516846

平均交叉验证分数:[np.float64(0.7863111111111111)]
整个数据集的拟合时间:[0.6124982833862305]


# SVC

In [11]:
# 太耗时，准确率也不高。放弃

# svc = make_pipeline(num_OHE_processor, SVC(class_weight='balanced', cache_size=8000, probability=True, random_state=0))
#
# param_grid = [
#     # 默认参数
#     #{},
#     # 太耗时，不搜索
#     # {
#     #     "svc__C": [0.1, 1, 10, 100],
#     #     "svc__gamma": ["scale", 0.001, 0.01, 0.1],
#     # }
# ]
#
# grid_search = GridSearchCV(svc, param_grid, scoring='accuracy', cv=3, n_jobs=-1)
# grid_search.fit(X_train, y_train)
#
# record('SVC', grid_search)


# 默认参数：
# '平均交叉验证分数': 0.8201333333333333, '整个数据集的拟合时间': 234.73922419548035
# 最佳参数：
# '平均交叉验证分数': np.float64(0.8481777777777777), '整个数据集的拟合时间': 42.788339614868164

# LinearSVC

In [12]:

linear_svc = make_pipeline(num_OHE_processor, LinearSVC(class_weight='balanced', random_state=0))

param_grid = [
    #默认参数
    #{},
    {
        'linearsvc__C': [0.001, 0.01, 0.1, 1, 10, 100],
        'linearsvc__tol': [1e-4, 1e-3]
    }
]
grid_search = GridSearchCV(linear_svc, param_grid, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

record('LinearSVC', grid_search)


# 默认参数：
# '平均交叉验证分数': np.float64(0.7867111111111111), '整个数据集的拟合时间': 0.09305548667907715
# 最佳参数：
# '平均交叉验证分数': np.float64(0.7867111111111111), '整个数据集的拟合时间': 0.08698034286499023

平均交叉验证分数:[np.float64(0.7867111111111111)]
整个数据集的拟合时间:[0.09941649436950684]


# DecisionTreeClassifier

In [13]:


decision_tree = make_pipeline(OE_processor, DecisionTreeClassifier(random_state=0, class_weight="balanced"))

param_grid = [
    #默认参数
    #{},
    {
        'decisiontreeclassifier__criterion': ['gini', 'entropy'],  # 衡量分裂质量的标准
        'decisiontreeclassifier__max_depth': [None, 10, 15, 20, 30],  # 限制树深防止过拟合
        'decisiontreeclassifier__min_samples_split': [2, 5, 10, 20],  # 分裂内部节点所需的最小样本数
        'decisiontreeclassifier__min_samples_leaf': [1, 2, 5, 10],  # 叶节点所需的最小样本数
        'decisiontreeclassifier__ccp_alpha': [0.0, 0.001, 0.01]  # 剪枝参数，是提升决策树泛化能力的关键
    }
]

grid_search = GridSearchCV(decision_tree, param_grid, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

record('DecisionTreeClassifier', grid_search)

# 默认参数：
# '平均交叉验证分数': 0.8424888888888888, '整个数据集的拟合时间': 0.19455647468566895
# 最佳参数：
# '平均交叉验证分数': 0.8424888888888888, '整个数据集的拟合时间': 0.19671106338500977

平均交叉验证分数:[np.float64(0.8424888888888888)]
整个数据集的拟合时间:[0.28861474990844727]


# KNeighborsClassifier

In [14]:

knn = make_pipeline(num_OHE_processor, KNeighborsClassifier())

param_grid = [
    #默认参数
    #{},
    {
        'kneighborsclassifier__n_neighbors': [5, 9, 15, 21],
        'kneighborsclassifier__weights': ['uniform', 'distance'],
        'kneighborsclassifier__metric': ['euclidean', 'manhattan']
    }
]

grid_search = GridSearchCV(
    knn,
    param_grid,
    scoring='accuracy',
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
record('KNeighborsClassifier', grid_search)

# 默认参数：
# '平均交叉验证分数': 0.8663555555555554, '整个数据集的拟合时间': 0.04407334327697754
# 最佳参数：
# '平均交叉验证分数': 0.8742666666666666, '整个数据集的拟合时间': 0.03929591178894043

平均交叉验证分数:[np.float64(0.8742666666666666)]
整个数据集的拟合时间:[0.0415949821472168]


C:\Users\Student\AppData\Roaming\Python\Python314t\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [0.86635556 0.86617778 0.87084444 0.87088889 0.87302222 0.87288889
 0.87293333 0.87302222        nan 0.86791111        nan 0.8712
        nan 0.87311111        nan 0.87426667]
  warnings.warn(


# 集成学习Bagging

## RandomForestClassifier

In [15]:


random_forest = make_pipeline(OHE_processor, RandomForestClassifier(class_weight="balanced", random_state=0))

param_grid = [
    #默认参数
    # {},
    {
        'randomforestclassifier__n_estimators': [200, 500],
        'randomforestclassifier__max_depth': [15, 25, None],
        'randomforestclassifier__min_samples_split': [2, 10],
    }
]

grid_search = GridSearchCV(random_forest, param_grid, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

record('RandomForestClassifier', grid_search)

# 默认参数：
# '平均交叉验证分数': 0.8818222222222223, '整个数据集的拟合时间': 2.1214535236358643
# 最佳参数：
# '平均交叉验证分数': 0.8812, '整个数据集的拟合时间': 4.181504964828491


平均交叉验证分数:[np.float64(0.8812)]
整个数据集的拟合时间:[4.0943827629089355]


# 集成学习Boosting

## AdaBoostClassifier

In [16]:

base_est = DecisionTreeClassifier(max_depth=1, class_weight="balanced")
ada = make_pipeline(OE_processor, AdaBoostClassifier(estimator=base_est, random_state=0))

param_grid = [
    #默认参数
    #{},
    {
        'adaboostclassifier__estimator__max_depth': [1, 2],
        'adaboostclassifier__n_estimators': [100, 300, 500],
        'adaboostclassifier__learning_rate': [0.05, 0.1, 0.5, 1.0],
    }
]

grid_search = GridSearchCV(
    estimator=ada,
    param_grid=param_grid,
    scoring='accuracy',
    n_jobs=-1,  # 榨干 i5-13500H 的性能
)

grid_search.fit(X_train, y_train)

record('AdaBoostClassifier', grid_search)

# 默认参数：
# '平均交叉验证分数': 0.7113777777777777, '整个数据集的拟合时间': 0.20016050338745117
# 最佳参数：
# '平均交叉验证分数': 0.7678666666666667, '整个数据集的拟合时间': 0.23143911361694336

平均交叉验证分数:[np.float64(0.7678666666666667)]
整个数据集的拟合时间:[0.21797609329223633]


## HistGradientBoostingClassifier

In [17]:

HGBT = make_pipeline(
    num_OE_processor,
    HistGradientBoostingClassifier(
        categorical_features=['cat__' + s for s in categorical_features.tolist()],
        class_weight="balanced",
        random_state=0
    )
)
HGBT.set_output(transform="pandas")
param_grid = [
    #默认参数
    #{},
    {
        'histgradientboostingclassifier__learning_rate': [0.05, 0.1, 0.2],
        'histgradientboostingclassifier__max_iter': [100, 200],
        'histgradientboostingclassifier__max_leaf_nodes': [15, 31, 63],
        'histgradientboostingclassifier__min_samples_leaf': [20, 50],
    }
]

grid_search = GridSearchCV(
    estimator=HGBT,
    param_grid=param_grid,
    scoring='accuracy',
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

record('HistGradientBoostingClassifier', grid_search)

# 默认参数：
# '平均交叉验证分数': 0.8288888888888888, '整个数据集的拟合时间': 0.3008120059967041
# 最佳参数：
# '平均交叉验证分数': 0.8380444444444445, '整个数据集的拟合时间': 0.39797115325927734

平均交叉验证分数:[np.float64(0.8380444444444445)]
整个数据集的拟合时间:[3.0152788162231445]


# 集成学习Stacking StackingClassifier

In [18]:
results = results.sort_values(['平均交叉验证分数'], ascending=False).reset_index(drop=True)
results

,估计器,估计器实例,平均交叉验证分数,整个数据集的拟合时间
0,RandomForestClassifier,"(ColumnTransformer(remainder='passthrough',\n ...",0.881200,4.094383
1,KNeighborsClassifier,"(ColumnTransformer(transformers=[('num', Stand...",0.874267,0.041595
2,DecisionTreeClassifier,"(ColumnTransformer(remainder='passthrough',\n ...",0.842489,0.288615
3,HistGradientBoostingClassifier,"(ColumnTransformer(transformers=[('num', Stand...",0.838044,3.015279
4,LinearSVC,"(ColumnTransformer(transformers=[('num', Stand...",0.786711,0.099416
5,LogisticRegression,"(ColumnTransformer(transformers=[('num', Stand...",0.786311,0.612498
6,AdaBoostClassifier,"(ColumnTransformer(remainder='passthrough',\n ...",0.767867,0.217976


In [19]:
# 使用同样的训练集训练3个估计器，再用同样的训练集评估StackingClassifier，存在数据泄露。
estimators = [
    ('RandomForestClassifier', results.loc[0, '估计器实例']),
    ('KNeighborsClassifier', results.loc[1, '估计器实例']),
    ('HistGradientBoostingClassifier', results.loc[3, '估计器实例'])
]

stacking = StackingClassifier(estimators=estimators)

scores = cross_val_score(stacking, X_train, y_train, scoring='accuracy', n_jobs=-1)
print(f'平均交叉验证分数: {scores.mean()}')


平均交叉验证分数: 0.8815555555555555
